# KNN

* Avaliação do CUML da Nvidia
* Avaliação da Geração de Código do Gemini-PRO
  * Python
  * CUDA
* Otimização em diversos cenários
  * K pequeno, muitas consultas
  * K grande, muitas consultas
  * Dataset Grande mais poucas Consultas
  * Versão com dois ou mais pontos por Thread
  * Uso de Registradores
  * Avaliação do estilo de código e PTX
  

# Integrantes do Grupo - 6 pontos
Nome | Matricula
--|--


# KNN com CUML

* Consulta de 100 amostras
* Dataset
  * 4 classes
  * 10 features
  * 1 Milhão de amostras
* K=5.

In [ ]:
#@title Python com CUML
import time
import cudf
sklearn
from cuml.datasets import make_classification
from cuml.neighbors import NearestNeighbors
import numpy as np

# Configurações do Dataset
n_samples = 1_000_000
n_features = 10
n_classes = 4

print(f"--- Iniciando processamento para {n_samples} amostras ---")

# --- PASSO 1: Gerar o Dataset ---
start_gen = time.time()
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_classes=n_classes,
    n_informative=8, # Definindo parâmetros para evitar erros de redundância
    random_state=42
)
end_gen = time.time()

print(f"Tempo para GERAR o dataset: {end_gen - start_gen:.4f} segundos")

# --- PASSO 2: Configurar e Treinar o KNN ---
# Usaremos o NearestNeighbors para busca de vizinhos próximos
knn = NearestNeighbors(n_neighbors=5)

start_fit = time.time()
knn.fit(X)
end_fit = time.time()

print(f"Tempo para INDEXAR (fit) o KNN: {end_fit - start_fit:.4f} segundos")

# --- PASSO 3: Executar uma Consulta ---
# Criando 100 amostras aleatórias para buscar os vizinhos
query_data = X[:100]

start_query = time.time()
distances, indices = knn.kneighbors(query_data)
end_query = time.time()

print(f"Tempo para CONSULTAR (100 amostras): {end_query - start_query:.4f} segundos")

--- Iniciando processamento para 1000000 amostras ---
Tempo para GERAR o dataset: 0.0318 segundos
Tempo para INDEXAR (fit) o KNN: 0.0015 segundos
Tempo para CONSULTAR (100 amostras): 0.0492 segundos


## Availiação

Considerando que temos que percorrer no mínimo o dataset todo (1 Milhão * 10 features) onde fazemos pelo menos uma subtração por feature e a redução = 20-30 Operações. Vamos supor 30, mais 5 operações para posicionar a amostra no grupo dos K=5 mais próximos, aproximando para 40 operações, podemos dizer:
* 40 operações
* 1 Milhão de amostras
* 100 consultas

Portanto $\frac{40 \cdot 100 \cdot 1M}{0.05s}= 80 Gops/s$  

### 1000 Amostras

$\frac{40 \cdot 1000 \cdot 1M}{0.0826s}= 484 Gops/s$

In [ ]:
# --- PASSO 4: Executar uma Consulta ---
# Criando 1000 amostras aleatórias para buscar os vizinhos
query_data = X[:1000]

start_query = time.time()
distances, indices = knn.kneighbors(query_data)
end_query = time.time()

print(f"Tempo para CONSULTAR (1000 amostras): {end_query - start_query:.4f} segundos")

Tempo para CONSULTAR (1000 amostras): 0.0826 segundos


### 10K amostras

$\frac{40 \cdot 10000 \cdot 1M}{0.40s}= 1000 Gops/s$

In [ ]:
# --- PASSO 5: Executar uma Consulta ---
# Criando 10000 amostras aleatórias para buscar os vizinhos
query_data = X[:10000]

start_query = time.time()
distances, indices = knn.kneighbors(query_data)
end_query = time.time()

print(f"Tempo para CONSULTAR (10000 amostras): {end_query - start_query:.4f} segundos")

Tempo para CONSULTAR (10000 amostras): 0.4013 segundos


### 100K amostras

$\frac{40 \cdot 100000 \cdot 1M}{3s}= 1300 Gops/s$

In [ ]:
# --- PASSO 5: Executar uma Consulta ---
# Criando 100000 amostras aleatórias para buscar os vizinhos
query_data = X[:100000]

start_query = time.time()
distances, indices = knn.kneighbors(query_data)
end_query = time.time()

print(f"Tempo para CONSULTAR (100000 amostras): {end_query - start_query:.4f} segundos")

Tempo para CONSULTAR (100000 amostras): 3.0273 segundos


Consultas | Tempo | Gops (40 ops)
--|--|--
100 | 0.05 | 80
1000 | 0.08 | 484
10000 | 0.4 | 1000
100000 | 3.0 | 1333


In [ ]:
#@title Versao com Interface Considerando somente 25 operações por ponto (quase metade dos anteriores)
import time
import ipywidgets as widgets
from IPython.display import display
from cuml.datasets import make_classification
from cuml.neighbors import NearestNeighbors

# 1. A Função Principal (Benchmark)
def run_knn_benchmark(n_features, k_neighbors, n_queries):
    n_samples = 1_000_000
    n_classes = 4

    print(f"--- Iniciando Execução ---")
    print(f"Amostras: {n_samples:,} | Atributos: {n_features} | Classes: {n_classes}")

    # Gerar os dados na GPU
    start_gen = time.time()
    n_info = max(2, n_features // 2)
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_classes=n_classes,
        n_informative=n_info,
        random_state=42
    )
    print(f"Tempo de geração: {time.time() - start_gen:.4f}s")

    print(f"\n--- Construindo o Índice KNN ---")
    print(f"Valor de K definido para: {k_neighbors}")
    start_fit = time.time()
    knn = NearestNeighbors(n_neighbors=k_neighbors)
    knn.fit(X)
    print(f"Tempo de indexação: {time.time() - start_fit:.4f}s")

    print(f"\n--- Executando Consultas ---")
    query_data = X[:n_queries]

    start_query = time.time()
    distances, indices = knn.kneighbors(query_data)
    end_query = time.time()

    query_time = end_query - start_query

    # Equações de Desempenho
    ops_per_pair = (2 * n_features) + k_neighbors
    total_ops = n_queries * n_samples * ops_per_pair

    if query_time > 0:
        ops_per_second = total_ops / query_time
        gflops = ops_per_second / 1e9
    else:
        ops_per_second = 0
        gflops = 0

    print(f"Número de Consultas: {n_queries:,}")
    print(f"Tempo EXATO da consulta: {query_time:.6f} segundos")
    print(f"Operações Totais Estimadas: {total_ops:,}")
    print(f"➔ Desempenho Alcançado: {gflops:.2f} GFLOPS\n")
    print("-" * 30)

# 2. Configurando a Interface de Usuário (Widgets)
style = {'description_width': '150px'}
layout = widgets.Layout(width='500px')

widget_features = widgets.IntSlider(min=2, max=100, step=1, value=10, description='Atributos (Features):', style=style, layout=layout)
widget_k = widgets.IntSlider(min=1, max=100, step=1, value=5, description='Valor de K:', style=style, layout=layout)
widget_queries = widgets.IntSlider(min=10, max=100000, step=10, value=100, description='Qtd de Consultas:', style=style, layout=layout)

# Criando o botão e a área de saída dos prints
button = widgets.Button(description="Executar Benchmark", button_style='success', icon='play', layout=widgets.Layout(width='200px', margin='10px 0 0 150px'))
output = widgets.Output()

# 3. Definindo a ação do clique do botão
def on_button_clicked(b):
    with output:
        output.clear_output() # Limpa a tela antes de rodar de novo
        run_knn_benchmark(widget_features.value, widget_k.value, widget_queries.value)

button.on_click(on_button_clicked)

# 4. Exibindo tudo na tela
ui = widgets.VBox([widget_features, widget_k, widget_queries, button])
display(ui, output)

Output()

# Versão PYCUDA

* Kernel em CUDA
* Chamada em Python

**IMPORTANTE**: precisa instalar no Colab, as vezes tem que fazer reset no runtime

In [ ]:
#@title Instalando
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.6 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=6466e2c8d0f3c4bcb5864a0f00db73d287dbed2d7d9098b6bea9d1468b513441
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


* Cada Thread avalia uma unica consulta e percorre o dataset completo
   * Tem reuso de memoria pois todas as threads vão ler a mesma amostra de dado
   * Se forem poucas consultas serão poucas threads....
   * Mantem o top-k com algoritmo de inserção
   * Usa unroll nos FOR uma vez que o K e número de features são conhecidos em tempo de compilação.



In [ ]:
#@title Primeira Versão
import time
import cupy as cp
import ipywidgets as widgets
from IPython.display import display
from cuml.datasets import make_classification

# 1. Código do Kernel CUDA em C++
# Para o CuPy, precisamos adicionar extern "C" antes do __global__
cuda_code = """
#define K 5
#define FEATURES 10

extern "C" __global__
void knn_kernel(const float* X_train, const float* X_query, float* out_dist, int* out_indices, int n_train, int n_queries) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n_queries) {
        float best_dist[K];
        int best_idx[K];

        #pragma unroll
        for(int i = 0; i < K; i++) {
            best_dist[i] = 3.402823466e+38F; // Máximo float
            best_idx[i] = -1;
        }

        float q[FEATURES];
        #pragma unroll
        for(int f = 0; f < FEATURES; f++) {
            q[f] = X_query[tid * FEATURES + f];
        }

        for(int i = 0; i < n_train; i++) {
            float dist = 0.0f;

            #pragma unroll
            for(int f = 0; f < FEATURES; f++) {
                float diff = q[f] - X_train[i * FEATURES + f];
                dist += diff * diff;
            }

            if (dist < best_dist[K-1]) {
                best_dist[K-1] = dist;
                best_idx[K-1] = i;

                for(int j = K-2; j >= 0; j--) {
                    if (best_dist[j+1] < best_dist[j]) {
                        float tmp_d = best_dist[j];
                        best_dist[j] = best_dist[j+1];
                        best_dist[j+1] = tmp_d;

                        int tmp_i = best_idx[j];
                        best_idx[j] = best_idx[j+1];
                        best_idx[j+1] = tmp_i;
                    } else {
                        break;
                    }
                }
            }
        }

        #pragma unroll
        for(int i = 0; i < K; i++) {
            out_dist[tid * K + i] = best_dist[i];
            out_indices[tid * K + i] = best_idx[i];
        }
    }
}
"""

# Compila o Kernel nativamente pelo CuPy
knn_kernel = cp.RawKernel(cuda_code, 'knn_kernel')

# 2. Função Principal
def run_cupy_raw_kernel(n_queries):
    n_samples = 1_000_000
    n_features = 10
    k_neighbors = 5

    print(f"--- 1. Gerando Dataset com cuML ---")
    start_gen = time.time()

    # O cuML já retorna arrays compatíveis com CuPy/RMM na GPU!
    X_train, _ = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_classes=4,
        n_informative=5,
        random_state=42
    )

    # Garantimos que é float32 e pegamos as queries direto da GPU
    X_train = cp.asarray(X_train, dtype=cp.float32)
    X_query = X_train[:n_queries]

    print(f"Tempo de geração (Zero Cópias para CPU): {time.time() - start_gen:.4f}s\n")

    print(f"--- 2. Executando Kernel Customizado via CuPy ---")
    # Aloca matrizes de saída diretamente na GPU
    out_dist = cp.empty((n_queries, k_neighbors), dtype=cp.float32)
    out_idx = cp.empty((n_queries, k_neighbors), dtype=cp.int32)

    # Configuração de Blocos e Threads
    threads_per_block = 128
    blocks_per_grid = (n_queries + threads_per_block - 1) // threads_per_block

    # Eventos para medir tempo estritamente na GPU
    start_event = cp.cuda.Event()
    end_event = cp.cuda.Event()

    start_event.record()

    # Chamada do Kernel
    knn_kernel(
        (blocks_per_grid,), (threads_per_block,),
        (X_train, X_query, out_dist, out_idx, cp.int32(n_samples), cp.int32(n_queries))
    )

    end_event.record()
    end_event.synchronize()

    # O tempo retornado por get_elapsed_time é em milissegundos
    query_time = cp.cuda.get_elapsed_time(start_event, end_event) / 1000.0

    # Cálculo de Performance
    ops_per_pair = (2 * n_features) + k_neighbors
    total_ops = n_queries * n_samples * ops_per_pair

    gflops = (total_ops / query_time) / 1e9 if query_time > 0 else 0

    print(f"Número de Consultas: {n_queries:,} | K: {k_neighbors} | Atributos: {n_features}")
    print(f"Tempo EXATO da consulta: {query_time:.6f} segundos")
    print(f"➔ Desempenho do Seu Kernel: {gflops:.2f} GFLOPS\n")
    print("-" * 40)

# 3. Interface de Usuário
style = {'description_width': '150px'}
widget_queries = widgets.IntSlider(min=10, max=10000, step=10, value=100, description='Qtd de Consultas:', style=style, layout=widgets.Layout(width='500px'))
button = widgets.Button(description="Executar CuPy Kernel", button_style='primary', icon='cogs', layout=widgets.Layout(width='200px', margin='10px 0 0 150px'))
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        run_cupy_raw_kernel(widget_queries.value)

button.on_click(on_button_clicked)
display(widgets.VBox([widget_queries, button]), output)

Output()

## Desempenho

Amostra | tempo | Gops (com 25 ops)
--|--|--
10000 | 0.41 | 600

Podemos observar que foi o mesmo tempo da versão CUML da Nvidia.


# Versão CUDA completa CPU/GPU

* Mesmo Kernel que a versão Pycuda


In [ ]:
#@title CUDA CPU/GPU
%%writefile knn.cu
#include <stdio.h>
#include <stdlib.h>
#include <float.h>
#include <time.h>

// Definindo as constantes em tempo de compilação para otimização
#define FEATURES 10
#define K 10
#define N_TRAIN 1000000
#define N_QUERIES 15000
#define N_CLASSES 4

// --- KERNEL CUDA ---
__global__ void knn_kernel(const float* X_train, const float* X_query, float* out_dist, int* out_indices, int n_train, int n_queries) {
    // Cada thread será responsável por UMA consulta
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n_queries) {
        float best_dist[K];
        int best_idx[K];

        // Inicializa com distância máxima
        #pragma unroll
        for(int i = 0; i < K; i++) {
            best_dist[i] = FLT_MAX;
            best_idx[i] = -1;
        }

        // Carrega os atributos da consulta para os registradores locais da thread
        float q[FEATURES];
        #pragma unroll
        for(int f = 0; f < FEATURES; f++) {
            q[f] = X_query[tid * FEATURES + f];
        }

        // Loop por todas as amostras de treinamento
        for(int i = 0; i < n_train; i++) {
            float dist = 0.0f;

            // Calcula a distância Euclidiana ao quadrado
            #pragma unroll
            for(int f = 0; f < FEATURES; f++) {
                float diff = q[f] - X_train[i * FEATURES + f];
                dist += diff * diff;
            }

            // Atualiza o Top-K se encontrar uma distância menor
            if (dist < best_dist[K-1]) {
                best_dist[K-1] = dist;
                best_idx[K-1] = i;

                // Insertion sort simples para manter os vizinhos ordenados
                for(int j = K-2; j >= 0; j--) {
                    if (best_dist[j+1] < best_dist[j]) {
                        float tmp_d = best_dist[j];
                        best_dist[j] = best_dist[j+1];
                        best_dist[j+1] = tmp_d;

                        int tmp_i = best_idx[j];
                        best_idx[j] = best_idx[j+1];
                        best_idx[j+1] = tmp_i;
                    } else {
                        break;
                    }
                }
            }
        }

        // Grava os resultados de volta na memória global
        #pragma unroll
        for(int i = 0; i < K; i++) {
            out_dist[tid * K + i] = best_dist[i];
            out_indices[tid * K + i] = best_idx[i];
        }
    }
}

// --- CÓDIGO HOST (CPU) ---
int main() {
    printf("--- Iniciando KNN em CUDA/C PURO ---\n");
    printf("Amostras: %d | Consultas: %d | Atributos: %d | K: %d\n", N_TRAIN, N_QUERIES, FEATURES, K);

    // 1. Alocação de Memória no Host (CPU)
    size_t train_size = N_TRAIN * FEATURES * sizeof(float);
    size_t query_size = N_QUERIES * FEATURES * sizeof(float);
    size_t dist_size = N_QUERIES * K * sizeof(float);
    size_t idx_size = N_QUERIES * K * sizeof(int);

    float *h_X_train = (float*)malloc(train_size);
    float *h_X_query = (float*)malloc(query_size);
    float *h_out_dist = (float*)malloc(dist_size);
    int *h_out_indices = (int*)malloc(idx_size);

    // 2. Gerando os dados sintéticos
    srand(42);
    for (int i = 0; i < N_TRAIN * FEATURES; i++) {
        h_X_train[i] = (float)rand() / RAND_MAX;
    }
    for (int i = 0; i < N_QUERIES * FEATURES; i++) {
        h_X_query[i] = (float)rand() / RAND_MAX;
    }

    // 3. Alocação de Memória no Device (GPU)
    float *d_X_train, *d_X_query, *d_out_dist;
    int *d_out_indices;
    cudaMalloc(&d_X_train, train_size);
    cudaMalloc(&d_X_query, query_size);
    cudaMalloc(&d_out_dist, dist_size);
    cudaMalloc(&d_out_indices, idx_size);

    // 4. Copiando dados da CPU para a GPU
    cudaMemcpy(d_X_train, h_X_train, train_size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_X_query, h_X_query, query_size, cudaMemcpyHostToDevice);

    // 5. Configuração do Kernel e Medição de Tempo via CUDA Events
    int threadsPerBlock = 128; // Múltiplo de 32 (Warp size)
    int blocksPerGrid = (N_QUERIES + threadsPerBlock - 1) / threadsPerBlock;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    printf("\nLançando o Kernel CUDA...\n");
    cudaEventRecord(start);

    // Chamada do Kernel
    knn_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_X_train, d_X_query, d_out_dist, d_out_indices, N_TRAIN, N_QUERIES);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    // 6. Verificação de Tempo e GFLOPS
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    float seconds = milliseconds / 1000.0f;

    // Fórmulas de desempenho
    long long ops_per_pair = (2 * FEATURES) + K;
    long long total_ops = (long long)N_QUERIES * (long long)N_TRAIN * ops_per_pair;
    float gflops = (total_ops / seconds) / 1e9;

    printf("Tempo EXATO da consulta: %f segundos\n", seconds);
    printf("Operações Totais Estimadas: %lld\n", total_ops);
    printf("➔ Desempenho Alcançado: %.2f GFLOPS\n", gflops);
    printf("----------------------------------------\n");

    // 7. Limpeza da memória
    cudaFree(d_X_train); cudaFree(d_X_query);
    cudaFree(d_out_dist); cudaFree(d_out_indices);
    free(h_X_train); free(h_X_query); free(h_out_dist); free(h_out_indices);

    return 0;
}

Overwriting knn.cu


## Desempenho

Para um conjunto com 10 milhões de amostras
* 10 features
* K=5
* Gops considerando 25 Ops

Amostras | Tempo | Gops/s
--|--|--
500k | 115 | 1087

In [ ]:
# Otimização de compilação flag -O3 para velocidade máxima da CPU e GPU
!nvcc knn.cu -o knn_exec -O3

# Executa o binário gerado
!nvprof ./knn_exec

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
--- Iniciando KNN em CUDA/C PURO ---
Amostras: 1000000 | Consultas: 15000 | Atributos: 10 | K: 10
==11992== NVPROF is profiling process 11992, command: ./knn_exec

Lançando o Kernel CUDA...
Tempo EXATO da consulta: 0.402038 segundos
Operações Totais Estimadas: 450000000000
➔ Desempenho Alcançado: 1119.30 GFLOPS
----------------------------------------
==11992== Profiling application: ./knn_exec
==11992== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.96%  401.80ms         1  401.80ms  401.80ms  401.80ms  knn_kernel(float const *, float const *, float*, int*, int, int)
                    2.04%  8.3581ms         2  4.1791ms  51.391us  8.3067ms  [CUDA memcpy HtoD]
      API calls:   67.04%  401.81ms         1  401.81ms  401.81ms  

In [ ]:
# Medir Registradores
!nvcc -arch=sm_75 -Xptxas -v knn.cu -o knn_exec

ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z10knn_kernelPKfS0_PfPiii' for 'sm_75'
ptxas info    : Function properties for _Z10knn_kernelPKfS0_PfPiii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 50 registers, used 0 barriers, 392 bytes cmem[0]
ptxas info    : Compile time = 10.347 ms


In [ ]:
#@title Compilando para PTX
import subprocess
import re
import os
from IPython.display import display, HTML

# 1. Compilar o código CUDA gerando APENAS o arquivo PTX
# Usamos -ptx para gerar o assembly, mantendo as restrições que você pediu
print("Compilando sort_registers.cu para PTX...")
comando = "nvcc -ptx knn.cu -o knn.ptx -arch=sm_75"
subprocess.run(comando, shell=True, check=True)
print("Arquivo knn.ptx gerado com sucesso!\n")

# 2. Ler o arquivo PTX real gerado pelo compilador
with open("knn.ptx", "r") as file:
    ptx_content = file.read()

# 3. Criar um Syntax Highlighting básico para ficar bonito no HTML
# Escapar os sinais de maior e menor do HTML
ptx_html = ptx_content.replace('<', '&lt;').replace('>', '&gt;')

# Regex para colorir comentários (verde)
ptx_html = re.sub(r'(//.*)', r'<span style="color: #6a9955;">\1</span>', ptx_html)
# Regex para colorir diretivas que começam com ponto (rosa)
ptx_html = re.sub(r'(\.\w+)', r'<span style="color: #c586c0; font-weight: bold;">\1</span>', ptx_html)
# Regex para colorir registradores que começam com % (azul claro)
ptx_html = re.sub(r'(%[a-zA-Z0-9_]+)', r'<span style="color: #9cdcfe;">\1</span>', ptx_html)

# 4. Montar a estrutura HTML
html_template = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    .ptx-container {{
        font-family: 'Courier New', Courier, monospace;
        background-color: #1e1e1e;
        color: #d4d4d4;
        padding: 15px;
        border-radius: 8px;
        box-shadow: 0 4px 8px rgba(0,0,0,0.3);
    }}
    .ptx-header {{
        color: #569cd6;
        margin-top: 0;
        border-bottom: 1px solid #444;
        padding-bottom: 10px;
        font-weight: bold;
    }}
    .ptx-code {{
        height: 600px; /* Altura fixa com rolagem */
        overflow-y: auto;
        white-space: pre; /* Mantém a formatação do assembly */
        font-size: 13px;
        line-height: 1.4;
    }}
    /* Customizando a barra de rolagem */
    ::-webkit-scrollbar {{ width: 10px; }}
    ::-webkit-scrollbar-track {{ background: #1e1e1e; }}
    ::-webkit-scrollbar-thumb {{ background: #555; border-radius: 5px; }}
    ::-webkit-scrollbar-thumb:hover {{ background: #777; }}
</style>
</head>
<body>
    <div class="ptx-container">
        <h3 class="ptx-header">📜 Código PTX Real (sort_registers.ptx)</h3>
        <div class="ptx-code">{ptx_html}</div>
    </div>
</body>
</html>
"""

# 5. Renderizar o HTML no output da célula
display(HTML(html_template))

Compilando sort_registers.cu para PTX...
Arquivo knn.ptx gerado com sucesso!



# **Tarefas para KNN**

Medidas básicas:

* Número de registradores por thread
* Número de Instruções no PTX
    * Observar como o Top-K foi implementado com setp ou bra ?

## 1.Avaliação de número mínimo de amostras

Sabendo que a T4 tem 2540 cores distribuídos em 40 multiprocessadores e a 4060 tem 3072 em 48 SMs. Verificar qual é o tamanho das amostras, já que é uma amostra por thread para ter o desempenho limitado pelo processamento e não pela subutilização da GPU. Fazer um código que gera um gráfico do tempo de execução em função do número de amostras e com iwidget do GOps, onde voce pode ajustar o valor médio de operações por amostra.

ideia é ter um grafico para saber quantos threads preciso para usar bem minha GPU...

## 2.Instrumentar o código para medir o Gops

* fazer uma versão de depuração que mede o número de operações no código de subtração e multiplicacoes é 20 (euclidiano), mas o insert sort depende. Anotar o insert sort acumulativo para cada amostra e depois fazer um histograma dos Ops por amostra e um boxplot.

Observar o código PTX, a principio usa "MOVE" para ir deslocando os top-5 valores na "fila", mas em alto nivel voce pode contar quantas vezes executa a iteracao do WHILE para saber se faz 20 + ... operacoes (2 para cada iteracao)

No inicio deve fazer bastante, no final deve ir reduzindo...pois já "achou" os top-5 mais proximos .



## 3.Otimização por grupo

proposta aqui cada grupo explorar com uma visão DIFERENTE de estrategia. Pode ser que não melhore.

Para cada grupo escolher uma estratégia e me avisar. Só precisa fazer uma por grupo.

**Estratégia 1**

Fazer uso de WARP. Cada WARP irá ler uma consulta. Supor que tenha 32 features (no lugar de 10). Assim cada thread do Warp faz o SUB e MUL , depois tem uma redução de distancia e apenas a thread 0 faz a insercao na fila. Comparar com o codigo de 1 amostra por thread (agora com 32 features).


**Estratégia 2**

Fazer uma estratégia de computação aproximada. Cada thread pega uma amostra. No inicio ele não descartar nada e vai juntando os top-k. Suponha que até I % das amostras. Por exemplo I=20, então nos 20% iniciais do dataset, não descartar nada.

Depois voce escolhe quanto ira descartar. D%. Suponha 30%, então dos 80% restantes, 30% o thread não é olhar. Será que ficara "30%" mais rapido ?
Será que perde precisão ?

Para isto voce tem que comparar a acuracia sem a aproximacao e com a aproximacao.

Testar diferentes valores de I e D. Os threads não podem cortar as mesmas amostras do dataset, faz o threard cortar ou não usando o numero dele, para ele cortar em lugares diferentes. Explicar sua proposta.



**Estratégia 3** Fazer uma versão que tenha desempenho para um número reduzido de consultas, por exemplo 128 consultas no dataset de 1milhão mas que seja rapido.
vamos usar potência de 2. Dataset tem 1 MEGA dados, voce quer 8K threads rodando com uma consulta compartilhada por thread.

$8K = 2^{13}$ divide por 128, então igual a 32. Então a thread irá olhar "1/32" do dataset.  1M= $2^{20}/2^5=2^{15}$ ou 32K amostras.

Pense em uma solução por WArp, t0 pega amostra0, t1 amostra1..., t0 pega amostra 32, etc... assim irá trabalhar alinhado. O "warp" então pegara' o top-k de cada thread e fazer um redução.







# DBSCAN


## Primeira Versão

* CUML
* Makeclassification para gerar o Dataset
* Não "detectou" as classes
* Métrica Inicial com 1 milhão de elementos, 10 Features
    * 100 segundos
    * 10*1M/100s = 0.1 Msamples/s
    

In [ ]:
#@title Versão CUML Gerado pelo Gemini PRO - Makeclassification
import time
import cupy as cp
from cuml.datasets import make_classification
from cuml.neighbors import NearestNeighbors
from cuml.cluster import DBSCAN

# --- 1. GERAÇÃO DO DATASET ---
n_samples = 1_000_000
n_features = 10
n_classes = 4

print("--- 1. Gerando Dataset com cuML ---")
start_gen = time.time()
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_classes=n_classes,
    n_informative=8,
    random_state=42
)
print(f"Tamanho do Dataset: {n_samples:,} linhas x {n_features} colunas")
print(f"Tempo de geração: {time.time() - start_gen:.4f} segundos\n")

# --- 2. EXECUÇÃO DO KNN (1000 Consultas) ---
n_queries = 1000
k_neighbors = 5
X_query = X[:n_queries] # Separando as 1000 amostras para consulta

print(f"--- 2. Executando KNN (K={k_neighbors}) para {n_queries} consultas ---")
# Fase de construção do índice (fit)
start_fit = time.time()
knn = NearestNeighbors(n_neighbors=k_neighbors)
knn.fit(X)
fit_time = time.time() - start_fit

# Fase de consulta
start_query = time.time()
distances, indices = knn.kneighbors(X_query)
query_time = time.time() - start_query

print(f"Tempo de Fit (KNN): {fit_time:.4f} segundos")
print(f"Tempo de Consulta (KNN): {query_time:.4f} segundos\n")

# --- 3. EXECUÇÃO DO DBSCAN ---
print("--- 3. Executando DBSCAN no dataset completo ---")
# O DBSCAN usa 'eps' (raio de vizinhança) e 'min_samples'.
# Em espaços de alta dimensão (10 features), distâncias tendem a ser maiores.
# Usaremos um eps um pouco maior do que o padrão (0.5) para evitar que tudo vire ruído.
eps_value = 3.0
min_samples_value = 15

start_dbscan = time.time()
dbscan = DBSCAN(eps=eps_value, min_samples=min_samples_value)
# O DBSCAN dispensa a variável 'y', ele tenta agrupar apenas usando as features (X)
labels_dbscan = dbscan.fit_predict(X)
dbscan_time = time.time() - start_dbscan

print(f"Tempo de Execução DBSCAN (1 milhão de amostras): {dbscan_time:.4f} segundos\n")

# --- 4. ANÁLISE DOS CLUSTERS vs CLASSES ORIGINAIS ---
print("--- 4. Análise de Resultados ---")
# Conta as labels únicas geradas pelo DBSCAN
unique_labels = cp.unique(labels_dbscan)

# O DBSCAN atribui a label '-1' para amostras que considera "Ruído" (outliers)
tem_ruido = -1 in unique_labels
n_clusters_dbscan = len(unique_labels) - (1 if tem_ruido else 0)

print(f"Classes originais geradas: {n_classes}")
print(f"Clusters detectados pelo DBSCAN: {n_clusters_dbscan}")

if tem_ruido:
    ruidos = cp.sum(labels_dbscan == -1)
    print(f"Amostras classificadas como Ruído (-1): {ruidos} de {n_samples:,}")

print("\n(Nota: O número de clusters no DBSCAN é altamente sensível aos parâmetros 'eps' e 'min_samples'. Em 10 dimensões, a Maldição da Dimensionalidade pode fazer com que pontos fiquem muito distantes, dificultando a formação exata de 4 clusters perfeitos sem um ajuste fino do eps).")


--- 1. Gerando Dataset com cuML ---
Tamanho do Dataset: 1,000,000 linhas x 10 colunas
Tempo de geração: 0.0313 segundos

--- 2. Executando KNN (K=5) para 1000 consultas ---
Tempo de Fit (KNN): 0.0008 segundos
Tempo de Consulta (KNN): 0.0826 segundos

--- 3. Executando DBSCAN no dataset completo ---
[2026-03-27 00:01:11.093] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 2492 -> 2147. Using the larger integer type might result in better performance
Tempo de Execução DBSCAN (1 milhão de amostras): 100.3555 segundos

--- 4. Análise de Resultados ---
Classes originais geradas: 4
Clusters detectados pelo DBSCAN: 1
Amostras classificadas como Ruído (-1): 1190 de 1,000,000

(Nota: O número de clusters no DBSCAN é altamente sensível aos parâmetros 'eps' e 'min_samples'. Em 10 dimensões, a Maldição da Dimensionalidade pode fazer com que pontos fiquem muito distantes, dificultando a formação exata de 4 clusters perfeitos sem um ajuste fino do eps).


## 4 Esferas

* Dataset fácil para verificação com 4 esferas espalhadas em 3D e os outros 7 features de ruído
* Versão com CUML
* 140s, 0.07 Msmaples/s

In [ ]:
#@title Versão 4 Esferas
import time
import cupy as cp
from cuml.datasets import make_blobs
from cuml.neighbors import NearestNeighbors
from cuml.cluster import DBSCAN

# --- 1. GERAÇÃO DO DATASET ESTRATÉGICO ---
n_samples = 1_000_000
n_features = 10
n_classes = 4

print("--- 1. Gerando Dataset com make_blobs (cuML) ---")

# Criando 4 centros de esferas, muito distantes uns dos outros nas 3 primeiras features.
# As 7 features restantes ficam no 0 (terão apenas valores pequenos de ruído/dispersão).
centros = cp.array([
    [ 150,  150,  150, 0, 0, 0, 0, 0, 0, 0], # Esfera 1
    [-150,  150, -150, 0, 0, 0, 0, 0, 0, 0], # Esfera 2
    [ 150, -150, -150, 0, 0, 0, 0, 0, 0, 0], # Esfera 3
    [-150, -150,  150, 0, 0, 0, 0, 0, 0, 0]  # Esfera 4
], dtype=cp.float32)

start_gen = time.time()
# cluster_std define o "tamanho" da esfera. Com std=5.0, os pontos ficam próximos ao centro.
X, y = make_blobs(
    n_samples=n_samples,
    n_features=n_features,
    centers=centros,
    cluster_std=5.0,
    random_state=42
)
print(f"Tamanho do Dataset: {n_samples:,} amostras x {n_features} atributos")
print(f"Tempo de geração: {time.time() - start_gen:.4f} segundos\n")


# --- 2. EXECUÇÃO DO KNN ---
n_queries = 1000
k_neighbors = 5
X_query = X[:n_queries]

print(f"--- 2. Executando KNN (K={k_neighbors}) para {n_queries} consultas ---")
start_fit = time.time()
knn = NearestNeighbors(n_neighbors=k_neighbors)
knn.fit(X)
fit_time = time.time() - start_fit

start_query = time.time()
distances, indices = knn.kneighbors(X_query)
query_time = time.time() - start_query

print(f"Tempo de Fit (Índice): {fit_time:.4f} segundos")
print(f"Tempo de Consulta Exata: {query_time:.4f} segundos\n")


# --- 3. EXECUÇÃO DO DBSCAN ---
print("--- 3. Executando DBSCAN no dataset completo ---")
# Como os centros estão separados por 300 unidades de distância (de 150 a -150),
# e as esferas têm raio em torno de 15 unidades (3x o std=5),
# um eps de 15.0 vai conectar todos os pontos de uma mesma esfera facilmente sem tocar na vizinha.
eps_value = 15.0
min_samples_value = 10

start_dbscan = time.time()
dbscan = DBSCAN(eps=eps_value, min_samples=min_samples_value)
labels_dbscan = dbscan.fit_predict(X)
dbscan_time = time.time() - start_dbscan

print(f"Tempo de Execução DBSCAN (1 milhão de pontos): {dbscan_time:.4f} segundos\n")


# --- 4. VALIDAÇÃO DOS CLUSTERS ---
print("--- 4. Análise de Resultados ---")
unique_labels = cp.unique(labels_dbscan)

tem_ruido = -1 in unique_labels
n_clusters_dbscan = len(unique_labels) - (1 if tem_ruido else 0)

print(f"Classes originais geradas (make_blobs): {n_classes}")
print(f"Clusters detectados pelo DBSCAN:        {n_clusters_dbscan}")

if tem_ruido:
    ruidos = cp.sum(labels_dbscan == -1)
    print(f"Amostras isoladas classificadas como Ruído (-1): {ruidos} de {n_samples:,}")
else:
    print("Nenhum ruído detectado! O agrupamento foi perfeito.")

print("-" * 40)

--- 1. Gerando Dataset com make_blobs (cuML) ---
Tamanho do Dataset: 1,000,000 amostras x 10 atributos
Tempo de geração: 0.5350 segundos

--- 2. Executando KNN (K=5) para 1000 consultas ---
Tempo de Fit (Índice): 0.0015 segundos
Tempo de Consulta Exata: 0.0825 segundos

--- 3. Executando DBSCAN no dataset completo ---
[2026-03-27 00:06:09.644] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 2492 -> 2147. Using the larger integer type might result in better performance
Tempo de Execução DBSCAN (1 milhão de pontos): 142.5091 segundos

--- 4. Análise de Resultados ---
Classes originais geradas (make_blobs): 4
Clusters detectados pelo DBSCAN:        4
Amostras isoladas classificadas como Ruído (-1): 6 de 1,000,000
----------------------------------------


# CUDA com Gemini PRO

In [ ]:
%%writefile dbscan_custom.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define N_SAMPLES 1000000
#define FEATURES 10
#define K_CLASSES 4
#define EPS 15.0f
#define MIN_PTS 10

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

// --- FUNÇÃO PARA GERAR RUÍDO GAUSSIANO (CPU) ---
float generate_gaussian(float mean, float std_dev) {
    float u1 = (float)rand() / RAND_MAX;
    float u2 = (float)rand() / RAND_MAX;
    if (u1 <= 0.0f) u1 = 1e-5f; // Evita log(0)
    float z0 = sqrtf(-2.0f * logf(u1)) * cosf(2.0f * M_PI * u2);
    return mean + z0 * std_dev;
}

// ==========================================================
// KERNELS CUDA PARA O DBSCAN
// ==========================================================

// Kernel 1: Identifica quem é "Core Point" (Ponto Central)
__global__ void find_cores(const float* X, int* is_core, int n, float eps_sq) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    int neighbors = 0;
    for (int j = 0; j < n; j++) {
        float dist_sq = 0.0f;
        #pragma unroll
        for (int f = 0; f < FEATURES; f++) {
            float diff = X[i * FEATURES + f] - X[j * FEATURES + f];
            dist_sq += diff * diff;
        }
        if (dist_sq <= eps_sq) {
            neighbors++;
        }
    }
    // Se tem vizinhos suficientes, é um Core Point
    is_core[i] = (neighbors >= MIN_PTS) ? 1 : 0;
}

// Função Auxiliar Device (GPU) para Union-Find
__device__ int find_root(int* parent, int i) {
    int root = i;
    while (root != parent[root]) root = parent[root];
    return root;
}

// Kernel 2: Conecta os Core Points usando Union-Find com Atomic Compare-And-Swap
__global__ void connect_cores(const float* X, int* is_core, int* parent, int n, float eps_sq) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n || !is_core[i]) return;

    for (int j = i + 1; j < n; j++) {
        if (is_core[j]) {
            float dist_sq = 0.0f;
            #pragma unroll
            for (int f = 0; f < FEATURES; f++) {
                float diff = X[i * FEATURES + f] - X[j * FEATURES + f];
                dist_sq += diff * diff;
            }

            if (dist_sq <= eps_sq) {
                // Mescla os dois conjuntos (Union)
                int root_i = find_root(parent, i);
                int root_j = find_root(parent, j);
                while (root_i != root_j) {
                    if (root_i < root_j) {
                        int old = atomicCAS(&parent[root_j], root_j, root_i);
                        if (old == root_j) break;
                        root_j = find_root(parent, old);
                    } else {
                        int old = atomicCAS(&parent[root_i], root_i, root_j);
                        if (old == root_i) break;
                        root_i = find_root(parent, old);
                    }
                }
            }
        }
    }
}

// Kernel 3: Achata a árvore do Union-Find para todos apontarem para o Root absoluto
__global__ void flatten_parents(int* parent, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int root = i;
        while (root != parent[root]) root = parent[root];
        parent[i] = root;
    }
}

// Kernel 4: Atribui os pontos de borda aos clusters e marca os ruídos
__global__ void assign_borders(const float* X, int* is_core, int* parent, int* labels, int n, float eps_sq) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    if (is_core[i]) {
        labels[i] = parent[i]; // Core points pegam o próprio grupo
    } else {
        int assigned = -1; // -1 significa ruído
        for (int j = 0; j < n; j++) {
            if (is_core[j]) {
                float dist_sq = 0.0f;
                #pragma unroll
                for (int f = 0; f < FEATURES; f++) {
                    float diff = X[i * FEATURES + f] - X[j * FEATURES + f];
                    dist_sq += diff * diff;
                }
                if (dist_sq <= eps_sq) {
                    assigned = parent[j]; // Entra no cluster do vizinho Core
                    break;
                }
            }
        }
        labels[i] = assigned;
    }
}


// ==========================================================
// CÓDIGO HOST (CPU)
// ==========================================================
int main() {
    printf("--- Iniciando Geração de Dados e DBSCAN em C/CUDA ---\n");

    size_t data_size = N_SAMPLES * FEATURES * sizeof(float);
    float* h_X = (float*)malloc(data_size);
    int* h_labels = (int*)malloc(N_SAMPLES * sizeof(int));

    // Centros das nossas 4 esferas perfeitas (nas 3 primeiras features)
    float centers[4][10] = {
        { 150,  150,  150, 0,0,0,0,0,0,0 },
        {-150,  150, -150, 0,0,0,0,0,0,0 },
        { 150, -150, -150, 0,0,0,0,0,0,0 },
        {-150, -150,  150, 0,0,0,0,0,0,0 }
    };

    srand(42);
    for (int i = 0; i < N_SAMPLES; i++) {
        int class_id = i % K_CLASSES; // Distribui igualmente as 4 classes
        for (int f = 0; f < FEATURES; f++) {
            // Std_dev de 5.0 gera a "esfera" em torno do centro
            h_X[i * FEATURES + f] = generate_gaussian(centers[class_id][f], 5.0f);
        }
    }
    printf("Gerado dataset com %d amostras.\n\n", N_SAMPLES);

    // Alocação na GPU
    float *d_X;
    int *d_is_core, *d_parent, *d_labels;
    cudaMalloc(&d_X, data_size);
    cudaMalloc(&d_is_core, N_SAMPLES * sizeof(int));
    cudaMalloc(&d_parent, N_SAMPLES * sizeof(int));
    cudaMalloc(&d_labels, N_SAMPLES * sizeof(int));

    // Inicializa arrays na GPU
    cudaMemcpy(d_X, h_X, data_size, cudaMemcpyHostToDevice);

    // Inicializa o array de pais (parent[i] = i) diretamente na VRAM
    int* h_init_parent = (int*)malloc(N_SAMPLES * sizeof(int));
    for(int i=0; i<N_SAMPLES; i++) h_init_parent[i] = i;
    cudaMemcpy(d_parent, h_init_parent, N_SAMPLES * sizeof(int), cudaMemcpyHostToDevice);
    free(h_init_parent);

    // Configuração de Threads
    int threads = 256;
    int blocks = (N_SAMPLES + threads - 1) / threads;
    float eps_sq = EPS * EPS; // Usamos distância ao quadrado para não calcular raiz na GPU

    // Medição de Tempo
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    printf("Executando DBSCAN na GPU...\n");
    cudaEventRecord(start);

    // Disparo dos 4 passos sequenciais do nosso DBSCAN
    find_cores<<<blocks, threads>>>(d_X, d_is_core, N_SAMPLES, eps_sq);
    cudaDeviceSynchronize();

    connect_cores<<<blocks, threads>>>(d_X, d_is_core, d_parent, N_SAMPLES, eps_sq);
    cudaDeviceSynchronize();

    flatten_parents<<<blocks, threads>>>(d_parent, N_SAMPLES);
    cudaDeviceSynchronize();

    assign_borders<<<blocks, threads>>>(d_X, d_is_core, d_parent, d_labels, N_SAMPLES, eps_sq);
    cudaDeviceSynchronize();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // Resgata os rótulos de volta para a CPU
    cudaMemcpy(h_labels, d_labels, N_SAMPLES * sizeof(int), cudaMemcpyDeviceToHost);

    // Contagem de Clusters na CPU
    int* unique_clusters = (int*)calloc(N_SAMPLES, sizeof(int));
    int count_clusters = 0;
    int count_noise = 0;

    for (int i = 0; i < N_SAMPLES; i++) {
        int label = h_labels[i];
        if (label == -1) {
            count_noise++;
        } else {
            // Como as labels são os IDs dos root points, usamos isso como chave
            if (unique_clusters[label] == 0) {
                unique_clusters[label] = 1;
                count_clusters++;
            }
        }
    }

    printf("\n--- RESULTADOS DO DBSCAN CUDA ---\n");
    printf("Tempo total de processamento: %.4f segundos\n", milliseconds / 1000.0f);
    printf("Classes baseadas nos centros: %d\n", K_CLASSES);
    printf("Clusters detectados:          %d\n", count_clusters);
    printf("Pontos ruidosos (-1):         %d\n", count_noise);
    if(count_clusters == 4 && count_noise == 0) {
        printf("SUCESSO: O algoritmo recriou as 4 esferas perfeitamente!\n");
    }

    // Limpeza
    cudaFree(d_X); cudaFree(d_is_core); cudaFree(d_parent); cudaFree(d_labels);
    free(h_X); free(h_labels); free(unique_clusters);

    return 0;
}

Overwriting dbscan_custom.cu


In [ ]:
!nvcc dbscan_custom.cu -o dbscan_exec -O3
!nvprof ./dbscan_exec

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
--- Iniciando Geração de Dados e DBSCAN em C/CUDA ---
Gerado dataset com 1000000 amostras.

==60469== NVPROF is profiling process 60469, command: ./dbscan_exec
Executando DBSCAN na GPU...

--- RESULTADOS DO DBSCAN CUDA ---
Tempo total de processamento: 70.8002 segundos
Classes baseadas nos centros: 4
Clusters detectados:          4
Pontos ruidosos (-1):         15
==60469== Profiling application: ./dbscan_exec
==60469== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   76.02%  53.8287s         1  53.8287s  53.8287s  53.8287s  connect_cores(float const *, int*, int*, int, float)
                   23.42%  16.5808s         1  16.5808s  16.5808s  16.5808s  find_cores(float const *, int*, int, float)
                    0.55%  388.01ms   

# Verificar outras implementações

* [fast db](https://github.com/l3lackcurtains/fast-cuda-gpu-dbscan?tab=readme-ov-file)
* [nesta pasta tem 4 artigos para ler e ver as ideias de soluções](https://drive.google.com/drive/folders/1m_3E9myylOhE-RAr3IOmt7Gb9Z0OygY1?usp=sharing)
     * Verificar se tem github
     * Baixar os datasets
     *  criar novos datasets
     * comparar com o baseline da Gemini-pro e Cuml para ver o desempenho dos códigos que tem github
     * Propor uma contribuição
         * Otimizar alguma parte do código
         * fazer o DBSCAN para varios "eps" e número mínimo de vizinhos simultaneamente
         * Propor nova ideia que possa fazer uso de computação aproximada ou nova estratégia de cálculo
         

## Primeira Otimização do Find Cores

* Executar com 10K threads, onde cada thread processa 100 elementos por vez, pulando de 10K em 10K no vetor de elementos para fazer reuso local da leitura do dado global em comparação com o dado local.
* Usar registradores e teste buscar 10 em 10, o ponto local para comparar com o global, promovendo reuso dentro da thread. Verificar o uso dos registradores para saber se pode ser de 20 em 20, ou outros fatores maiores.


## Segunda Otimizacao a ser pensada

* Merge para 100 pontos do Connect cores e o Find Cores para reduzir o tempo do Connect.
